# STEP 9 — 2단계 백본 비교

**2단계(병변 6종)는 지금까지 `resnet50` 하나로만 돌았습니다. 한 번도 비교한 적이 없습니다.**

1단계는 STEP 6 에서 `effnetv2_s` 로 바꿨지만, 그 결과를 2단계에 옮겨 적으면 안 됩니다.
같은 `photometric` 증강이 1단계에는 약이고 2단계에는 독이었습니다. 두 단계는 보는 것이 다릅니다.

## 이 노트북이 안 하는 것

| | 왜 |
|---|---|
| 1단계 재학습 | STEP 6·7 에서 `effnetv2_s` + `photometric` 으로 이미 정했습니다 |
| holdout 열기 | 후보를 고르는 데 holdout 을 쓰면 그 순간 오염됩니다. 05 가 엽니다 |
| 03 의 체크포인트 필요 | 여기서는 처음부터 학습합니다. `release` 를 붙일 필요가 없습니다 |

## 판정 기준 (실험 **전에** 못 박습니다 — 작업 규칙 2)

1. macro-F1 이 `resnet50` 보다 **+0.02 넘게** 높아야 교체 후보. 그 안이면 **기준선 유지**
2. 점수가 올라도 **배율 하락이 8%p 넘게 나빠지면** 탈락
3. 둘 다 만족하는 후보가 여럿이면 macro-F1 최고

규칙은 `experiments.backbone_report()` 에 있습니다 (노트북 셀 말고 — 작업 규칙 3).

> ⚠️ **val 최고점을 그냥 고르지 않습니다.** STEP 5·6 에서 그렇게 골랐다가
> holdout 에서 무너졌습니다.

## 필요한 Kaggle 입력

`dogskin-m25` 하나면 됩니다 (2단계는 m2.5 확정). `dogskin-full` 은 필요 없습니다.

In [ ]:
# ── 0. 환경 준비 (Colab / Kaggle 공통) ──────────────────────────
# 이 셀 하나가 리포 동기화 → 패키지 설치 → 환경 감지까지 다 합니다.
# 리포를 직접 다운로드하거나 드라이브에 올릴 필요 없습니다.
# 다시 실행하면 항상 최신 코드로 맞춰집니다 (로컬 수정은 덮어씁니다).
import os, sys, subprocess

REPO   = "https://github.com/gayeoniee/deeplearning_test.git"
NAME   = "deeplearning_test"
# ⚠️ 브랜치를 "main" 으로 **못 박으면 안 됩니다.** 아래 reset --hard 가
#    작업 브랜치를 통째로 덮어써서, 방금 만든 코드가 사라진 채로 몇 시간을
#    돌게 됩니다. 이미 리포 안에서 돌고 있으면 **지금 브랜치를 그대로 씁니다.**
#    바꾸려면 환경변수:  export DOG_SKIN_BRANCH=main
BRANCH = os.environ.get("DOG_SKIN_BRANCH", "")
_cwd   = os.getcwd()
if os.path.basename(_cwd) == NAME and os.path.isdir(os.path.join(_cwd, ".git")):
    DIR = _cwd            # 이미 리포 안에서 재실행 중 (중첩 clone 방지)
    if not BRANCH:
        BRANCH = subprocess.run(["git", "-C", DIR, "rev-parse", "--abbrev-ref", "HEAD"],
                                capture_output=True, text=True).stdout.strip() or "main"
else:
    # ⚠️ Kaggle 을 먼저 봅니다. Kaggle 이미지에도 /content 가 있어서
    #    /content 를 먼저 보면 Kaggle 세션인데 /content 에 clone 합니다.
    BASE = ("/kaggle/working" if os.path.isdir("/kaggle/working")
            else "/content" if os.path.isdir("/content") else _cwd)
    DIR = os.path.join(BASE, NAME)

BRANCH = BRANCH or "main"
if os.path.isdir(os.path.join(DIR, ".git")):
    # 이미 받아둔 경우: 최신으로 강제 동기화 (shallow clone 에서도 안전)
    subprocess.run(["git", "-C", DIR, "fetch", "--depth", "1", "origin", BRANCH], check=False)
    subprocess.run(["git", "-C", DIR, "reset", "--hard", f"origin/{BRANCH}"], check=False)
else:
    subprocess.run(["git", "clone", "-b", BRANCH, "--depth", "1", REPO, DIR], check=True)

os.chdir(DIR)
if DIR not in sys.path:
    sys.path.insert(0, DIR)

# ⚠️ 중요: 이미 import 된 src.* 는 파이썬이 캐시하고 있어서
#    파일을 갱신해도 옛날 코드가 그대로 쓰입니다. 캐시를 비웁니다.
for _m in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[_m]

print("작업 디렉터리:", os.getcwd())
print("코드 버전   :", subprocess.run(["git", "-C", DIR, "log", "--oneline", "-1"],
                                      capture_output=True, text=True).stdout.strip())

# 패키지 설치는 **uv 로 통일**합니다 (pip 보다 훨씬 빠릅니다).
# ⚠️ Colab/Kaggle 이미지에는 uv 가 없어서, uv 자체만 pip 로 한 번 받습니다.
#    --system = 가상환경을 새로 만들지 않고 이미 있는 파이썬에 그대로 설치.
#    (torch/numpy/pandas 는 이미 깔려 있으므로 여기서 안 건드립니다)
# albumentations 는 import 할 때마다 PyPI 에 버전 확인 요청을 보냅니다.
# Kaggle 은 외부 네트워크가 막혀 있어 타임아웃(2초)만 기다리다 끝납니다 — 꺼둡니다.
os.environ["NO_ALBUMENTATIONS_UPDATE"] = "1"

_PKGS = ["timm", "imagehash", "pyarrow", "grad-cam", "albumentations"]
_ok = False
if subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"],
                  check=False).returncode == 0:
    _ok = subprocess.run([sys.executable, "-m", "uv", "pip", "install", "-q",
                          "--system", *_PKGS], check=False).returncode == 0
if not _ok:
    print("[env] uv 로 설치하지 못해 pip 으로 대체합니다")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_PKGS], check=False)

# 한글 그래프 폰트 (Colab 기본에는 한글이 없어 □ 로 나옵니다)
_font = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if not os.path.exists(_font):
    subprocess.run(["apt-get", "install", "-y", "-qq", "fonts-nanum"], check=False)
try:
    import matplotlib.pyplot as plt, matplotlib.font_manager as fm
    fm.fontManager.addfont(_font)
    plt.rcParams["font.family"] = "NanumGothic"
    plt.rcParams["axes.unicode_minus"] = False
except Exception:
    pass

MY_NOTEBOOK_VERSION = "2026-08-25.1"   # ★ 이 셀(=이 .ipynb)의 버전

from src import env
from src.config import CFG, CLASSES, CLASS_KO
E = env.describe()
env.set_seed(42)

# ⚠️ 노트북 셀은 git pull 로 갱신되지 않습니다 (src/ 만 최신이 됩니다).
#    낡은 .ipynb 를 몇 시간 돌리고 나서 알게 되면 늦으므로 지금 확인합니다.
from src.config import NOTEBOOK_VERSION as _repo_nb
if MY_NOTEBOOK_VERSION != _repo_nb:
    print("\n" + "!" * 62)
    print(f"⚠️ 이 노트북이 낡았습니다 — 내 셀 {MY_NOTEBOOK_VERSION} / 리포 {_repo_nb}")
    print("   src/ 는 최신이지만 **셀 내용은 예전 것**입니다.")
    print("   GitHub 에서 notebooks/*.ipynb 를 다시 받아 Import 하세요:")
    print("   Kaggle → File → Import Notebook / Colab → 파일 → 노트 업로드")
    print("!" * 62 + "\n")
else:
    print(f"[nb] 노트북 최신 ({_repo_nb})")


## 0-b. 로컬에서 만든 데이터 불러오기

In [ ]:
# Drive 마운트는 **진짜 Colab VM** 에서만 시도합니다.
# ⚠️ Kaggle 에도 google.colab 패키지와 /content 가 있어서, 환경 판정을 잘못하면
#    Kaggle 에서 drive.mount() 를 부르고 NotImplementedError 로 죽습니다.
if env.can_mount_drive():
    env.mount_drive()
else:
    print(f"[env] {E.env} — Drive 마운트 없이 진행합니다")

# Colab: Drive 의 zip 해제 / Kaggle: /kaggle/input 의 풀린 폴더에 링크
env.load_prepared()

# 다른 환경에서 학습했다면 체크포인트를 가져옵니다 (Colab → Kaggle 이주)
# train.import_checkpoints("/kaggle/input/dogskin-ckpt")

_persist = env.persist_root()
if _persist is None:
    print("\n🚨 세션 밖 저장소가 없습니다 — 끊기면 체크포인트가 사라집니다. env.mount_drive() 확인.")
else:
    print(f"\n✅ 중단 대비 저장소: {_persist}")
    if E.env == "kaggle":
        print("   ⚠️ Kaggle 은 세션이 끝나면 /kaggle/working 이 사라질 수 있습니다.")
        print("      · 짧게 확인만 할 때  : 그냥 진행 (세션 안에서는 이어받기가 됩니다)")
        print("      · 긴 학습을 돌릴 때  : 우측 상단 [Save Version] →")
        print("                             'Save & Run All (Commit)' 로 돌리세요.")
        print("                             브라우저를 닫아도 끝까지 돌고, 출력이 보존됩니다.")
        print("      · 설정에 Persistence 항목이 보이면 'Files' 로 켜두면 더 안전합니다")

In [ ]:
# ── 1. 설정 ─────────────────────────────────────────────────────
# ⚠️ 여기 숫자를 바꾸면 아래 시간 추정도 같이 바뀝니다.
IMG_SIZE = 384          # 03 에서 실측으로 채택 (224 → 384 로 배율 하락 8~9%p 감소)
EPOCHS   = 25           # ★ 12 → 25 (STEP 9 재실행). 12 에서는 기준선 resnet50 이
                        #   마지막 에폭까지 계속 오르는 중이라 비교가 무효였습니다.
                        #   덜 학습된 기준선은 교란 검사에서 잃을 게 적어 배율 하락이
                        #   실제보다 낮게 나오고, 그러면 다른 백본이 부당하게 나빠 보입니다.
                        #   STEP 10 풀 데이터에서 resnet50 best 가 epoch 18 이었으므로
                        #   서브셋은 더 뒤일 수 있습니다. 조기 종료(patience=5)가 알아서 끊습니다.
SUBSET   = 0.55         # 학습셋만 55% (검증셋은 그대로). STEP 6 과 같은 조건
N_ROBUST = 2000         # 교란 검사 표본. ⚠️ 표본 수가 다르면 하락폭 비교 불가

# ── 비교할 백본 ────────────────────────────────────────────────
# ★ 한 번에 다 못 돌립니다. 두 가지 제약이 있습니다.
#
#   ① 기준선(resnet50)이 **같은 실행 안에** 있어야 합니다.
#      판정 규칙이 전부 "기준선 대비" 인데, 다른 세션 결과와 비교하면
#      실행 간 잡음(macro-F1 ±0.02 · 배율 하락 9.6%p)에 묻힙니다.
#   ② Kaggle 세션 9시간. convnextv2_base 혼자 에폭당 ~11분입니다
#      (resnet50 1.9분 / effnetv2_s 2.3분 — STEP 9 실측).
#
# 그래서 판(batch)을 나눠 돌립니다. 각 판에 기준선 비용(~48분)을 한 번씩 냅니다.
BATCH = "A"          # ← "A" / "B" / "C" 중 하나로 바꿔서 돌리세요

# CNN 은 해상도가 자유로워 384 로 맞춥니다. ViT 계열은 위치 임베딩·윈도우
# 크기가 고정이라 **자기 해상도**로 돌립니다 (멘토 피드백 8번).
#
# ⚠️ 해상도가 큰 모델이 유리한 게 아닙니다. 우리 크롭 짧은 변 중앙값이 242px 이고
#    384 미만을 74.6% 확대해서 쓰는 중입니다. eva02(336/448)는 없는 디테일을
#    만들어서 넣는 셈이라 순위를 뒤로 뒀습니다 — 해상도를 512 로 안 올린 것과 같은 이유.
BATCHES = {
    # STEP 9 다시 하기 (그때 기준선이 12에폭에서 안 끝나 판정이 무효였습니다)
    "A": [("resnet50", IMG_SIZE), ("effnetv2_s", IMG_SIZE),
          ("convnextv2_base", IMG_SIZE)],
    # 계층적 ViT + 대규모 사전학습. 둘 다 256px 라 우리 크롭(242px)과 맞습니다
    "B": [("resnet50", IMG_SIZE), ("swinv2_base", 256), ("siglip2_base", 256)],
    # 상한 확인용. 가장 무겁고 확대가 가장 심합니다
    "C": [("resnet50", IMG_SIZE), ("eva02_base", 336)],
}
MODELS = BATCHES[BATCH]

print(f"판 {BATCH} — 2단계 백본 {len(MODELS)}종 | {EPOCHS}에폭 | 학습셋 {SUBSET:.0%}")
for k, s in MODELS:
    print(f"   {k:<18} {s}px" + ("   ← 기준선" if k == "resnet50" else ""))
print("\n⚠️ 판마다 기준선이 다시 학습됩니다 — 판을 넘나들며 절대값을 비교하지 마세요.")
print("   판 안에서의 '기준선 대비' 만 유효합니다.")

In [ ]:
import gc, json
import torch
from src import labels, split, crop, models, experiments, train
from src.config import CLASSES, ADOPTED_STAGE2_CROP

env.require_gpu()
DEV = "cuda" if torch.cuda.is_available() else "cpu"

# ── 크롭은 채택값을 씁니다 ──────────────────────────────────────
# ⚠️ 예전에는 03 이 남긴 stage1_threshold.json 에서 읽었는데, Kaggle 은
#    노트북마다 세션이 달라 그 파일이 없습니다. 그러면 조용히 'm1.5' 로
#    떨어져서 **채택하지도 않은 크롭으로 3시간을 학습**하게 됩니다.
#    2단계 크롭은 STEP 4C 에서 확정됐으니 config 의 단일 출처를 씁니다.
BEST_CROP = ADOPTED_STAGE2_CROP
tags = crop.available_tags()
print(f"2단계 크롭: {BEST_CROP}  |  붙어 있는 태그: {tags}")
if BEST_CROP not in tags:
    raise SystemExit(
        f"❌ 채택 크롭 '{BEST_CROP}' 가 안 붙어 있습니다.\n"
        f"   붙어 있는 것: {tags}\n"
        "   Kaggle 이면 [Add Input] 으로 'dogskin-m25' 를 붙이세요.")

df = labels.load(env.work_root() / "manifests" / "manifest_final.parquet")
from src import stages
s2_all = stages.to_stage2(crop.switch_tag(df, BEST_CROP))
tr2, va2 = split.get_fold(s2_all, 0)
print(f"2단계  train {len(tr2):,} / val {len(va2):,}  (비교는 fold 0 고정)")

In [ ]:
# ── 2. 시작 전에 시간부터 재봅니다 (작업 규칙 3) ────────────────
# 3시간 돌린 뒤에 세션 한도를 넘겨 날리는 게 제일 아깝습니다.
# ★ (이름, 해상도) 쌍을 그대로 넘깁니다. 판 B 는 384(CNN)와 256(ViT)이 섞여
#   있는데, 전부 384 로 재면 ViT 는 만들어지는 해상도와 안 맞아 죽습니다.
experiments.estimate_runtime(MODELS, IMG_SIZE,
                             n_train=int(len(tr2) * SUBSET), epochs=EPOCHS,
                             device=DEV)
print("\n⚠️ Kaggle 세션 한도(9시간)를 넘길 것 같으면 위 MODELS 를 줄이고 다시 추정하세요.")
print("   [Save Version] → Save & Run All (Commit) 으로 돌려야 브라우저를 닫아도 남습니다.")

In [ ]:
# ── 3. 스윕 ─────────────────────────────────────────────────────
# ★ 이어받기: 끝난 모델은 train.fit 이 건너뜁니다. 세션이 죽으면 이 셀만 다시 돌리세요.
runs = []
for key, size in MODELS:
    try:
        r = experiments.train_and_measure(
            s2_all, stage=2, img_size=size, crop_tag=BEST_CROP, device=DEV,
            epochs=EPOCHS, model_name=key, finetune="moderate", aug="default",
            subset_frac=SUBSET, n_robust=N_ROBUST,
            measure_robust=True, measure_blur=False)
        runs.append(r)
    except torch.cuda.OutOfMemoryError:
        print(f"❌ {key}: VRAM 부족 — 건너뜁니다 (해상도를 낮춰 다시 시도해 보세요)")
    except Exception as exc:                                    # noqa: BLE001
        print(f"❌ {key}: {type(exc).__name__}: {exc}")
    finally:
        gc.collect()
        if DEV == "cuda":
            torch.cuda.empty_cache()

# ⚠️ 전부 실패해도 다음 셀로 넘어가면 "결과 0개" 를 몇 시간 뒤에 알게 됩니다.
#    실제로 한 번 그랬습니다 — 여기서 멈춥니다.
if not runs:
    raise SystemExit("❌ 성공한 실행이 0개입니다. 위 오류 메시지를 확인하세요.")
if not any(r["model_name"] == "resnet50" for r in runs):
    print("⚠️ 기준선 resnet50 이 실패했습니다 — 상대 비교가 약해집니다.")
print(f"\n✅ {len(runs)}/{len(MODELS)}개 완료")

## 4. 판정

규칙은 `src/experiments.py` 에 있습니다. 결과를 보고 기준을 고르지 않기 위해서입니다.

In [ ]:
verdict = experiments.backbone_report(runs, base_model="resnet50")

In [ ]:
# ── 5. 기록 남기기 ──────────────────────────────────────────────
out = env.work_root() / "reports"
out.mkdir(parents=True, exist_ok=True)
payload = {
    "step": "STEP9_2단계_백본비교",
    "batch": BATCH,                      # ★ 어느 판이었는지 — 판을 넘어 비교 금지
    "crop": BEST_CROP, "img_size": IMG_SIZE, "epochs": EPOCHS,
    "subset_frac": SUBSET, "n_robust": N_ROBUST,
    "runs": [{k: v for k, v in r.items()
              if isinstance(v, (str, int, float, bool, type(None)))} for r in runs],
    "picked": verdict.get("best", {}).get("model_name"),
}
fp = out / f"step9_backbone_{BATCH}.json"          # ★ 판마다 다른 파일로
fp.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
print("저장:", fp)
print("\n다음 —")
print(f"  1) 다른 판(B/C)도 돌릴 거면 BATCH 를 바꿔 다시 Commit")
print(f"  2) 판 {BATCH} 채택 후보 '{payload['picked']}' 를 06 에서 풀 학습으로 확인")
print("     (06 이 학습 + holdout 을 한 세션에서 끝냅니다 — 인계 없음)")
print("  ⚠️ 여기 숫자는 서브셋 결과입니다. 보고서에는 풀 학습 숫자를 쓰세요.")

---

## 다음에 해볼 수 있는 것 (지금은 안 함)

| | 왜 미루나 |
|---|---|
| 앙상블 (상위 2~3개 logit 평균) | 백본이 정해진 뒤에. 지금 하면 무엇이 기여했는지 안 보입니다 |
| Mixup / CutMix | 증강 축은 STEP 4B 에서 닫았습니다. 백본이 바뀌면 다시 열어볼 수 있습니다 |
| 5-fold 교차검증 | A6·A4 recall 의 측정 정밀도 문제(±0.085)를 푸는 카드. 최종 보고용 |

## 관련 기록

- [`docs/results/STEP4C_크롭비교_실측.md`](../docs/results/STEP4C_크롭비교_실측.md) — 2단계 크롭을 m2.5 로 확정한 근거
- [`docs/results/STEP6_1단계_2x2_실측.md`](../docs/results/STEP6_1단계_2x2_실측.md) — 1단계 백본을 바꾼 근거
- [`docs/results/STEP8_1단계교체_holdout_실측.md`](../docs/results/STEP8_1단계교체_holdout_실측.md) — 현재 기준선